<a href="https://colab.research.google.com/github/SiddardhaKoduri/AI_Powered_Duplicate_Account_Detection/blob/main/ClaimIntel_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!pip install -q \
streamlit \
pyngrok \
faiss-cpu \
sentence-transformers \
langchain \
langchain-community \
langchain-text-splitters \
pypdf \
openai

In [41]:
!pip install -U langchain-text-splitters

In [42]:
import os
import pickle
import faiss

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI

API Section

In [43]:
NVIDIA_API_KEY = "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"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

In [44]:
response = client.chat.completions.create(
    model="meta/llama-3.3-70b-instruct",
    messages=[
        {
            "role": "user",
            "content": "Introduce yourself in one sentence."
        }
    ],
    temperature=0.2,
    max_tokens=100
)

print(response.choices[0].message.content)

I'm an artificial intelligence model known as Llama, which stands for "Large Language Model Meta AI," and I'm here to assist and provide information to users through text-based conversations.


Upload pdf

In [45]:
from google.colab import files

uploaded = files.upload()

Saving MultiDomain_Insurance_Compendium.pdf to MultiDomain_Insurance_Compendium (2).pdf


Load PDFs

In [46]:
documents = []

for file in uploaded.keys():
    loader = PyPDFLoader(file)
    documents.extend(loader.load())

print("Number of pages loaded:", len(documents))

Number of pages loaded: 23


Spliting Documents into Chunks

In [47]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 81


In [48]:
# Create metadata with page numbers

import pickle

metadata = []

for chunk in chunks:
    metadata.append(
        {
            "text": chunk.page_content,
            "page": chunk.metadata["page"] + 1
        }
    )

print(metadata[0])

{'text': 'BHARAT UNIVERSAL INSURANCE\n LTD.\n Multi-Domain Insurance Policy Compendium\n Banking Insurance · Health Insurance · Motor Vehicle Insurance · Home & Property Insurance · Travel\n Insurance\nDocument Reference\nBUIL-POL-2025-0047\nVersion\n3.2 (Revised Edition)\nEffective Date\n01 April 2025\nReview Date\n31 March 2026\nRegulatory Filing\nIRDAI/HLT/BUIL/P-T/V.III/447/2025\nJurisdiction\nRepublic of India\nThis compendium consolidates the terms, conditions, coverage schedules, exclusions, and claims procedures', 'page': 1}


In [49]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generate Embeddings

In [50]:
texts = [chunk.page_content for chunk in chunks]
texts = [item["text"] for item in metadata]
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(81, 384)


FAISS Index

In [51]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

Vectors stored: 81


Saveing the Index

In [52]:
faiss.write_index(index, "index.faiss")

with open("metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Saved successfully!")

Saved successfully!


Retrieval + Llama 3.3 Function

In [53]:
def ask_claimintel(question):

    # Create query embedding
    query_embedding = embedding_model.encode([question])

    # Retrieve top 5 chunks
    D, I = index.search(query_embedding, 5)

    # Combine retrieved text
    context = "\n\n".join([texts[i] for i in I[0]])

    prompt = f"""
You are ClaimIntel, an intelligent insurance assistant.

Use ONLY the information provided below.

Context:
{context}

Question:
{question}

Provide a concise and accurate answer.
"""

    response = client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1024
    )

    return response.choices[0].message.content

rag_engine

In [54]:
%%writefile rag_engine.py

import pickle
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# ================= NVIDIA API =================
NVIDIA_API_KEY = "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"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

# ================= Embedding Model =================
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# ================= Load FAISS Index =================
index = faiss.read_index("index.faiss")

# ================= Load Metadata =================
with open("metadata.pkl", "rb") as f:
    metadata = pickle.load(f)


def ask_claimintel(question, chat_history=""):

    try:

        # Generate query embedding
        query_embedding = embedding_model.encode([question])

        # Retrieve top 5 chunks
        D, I = index.search(query_embedding, 5)

        # Get retrieved chunks
        retrieved_chunks = [metadata[i] for i in I[0]]

        # Build context
        context = "\n\n".join(
            chunk["text"] for chunk in retrieved_chunks
        )

        # Collect unique page numbers
        source_pages = sorted(
            set(
                chunk["page"]
                for chunk in retrieved_chunks
            )
        )

        # Prompt
        prompt = f"""
You are ClaimIntel, an intelligent insurance policy assistant.

Use ONLY the information provided below.

Previous Conversation:
{chat_history}

Retrieved Context:
{context}

Current Question:
{question}

Instructions:
- Answer accurately and clearly.
- Use only the provided context.
- If the answer is not present, say "I couldn't find that information in the policy documents."
"""

        response = client.chat.completions.create(
            model="meta/llama-3.3-70b-instruct",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=1024
        )

        answer = response.choices[0].message.content

        return {
            "answer": answer,
            "pages": list(source_pages)
        }

    except Exception as e:

        return {
            "answer": f"⚠️ Error: {str(e)}",
            "pages": []
        }

Overwriting rag_engine.py


In [55]:
from rag_engine import ask_claimintel

result = ask_claimintel(
    "what is the premium policy?"
)

print(result)

{'answer': "I couldn't find that information in the policy documents.", 'pages': [9, 12, 14, 21]}


In [56]:
import importlib
import rag_engine

importlib.reload(rag_engine)

from rag_engine import ask_claimintel

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

App UI

In [57]:
%%writefile app.py

import streamlit as st
from rag_engine import ask_claimintel

# ---------------- Page Config ----------------
st.set_page_config(
    page_title="ClaimIntel Copilot",

    layout="wide"
)

# ← ADD THIS BELOW
st.markdown("""
<style>

/* ========= BACKGROUND ========= */

.stApp{
background:
radial-gradient(circle at top left,#312E81 0%,#0B1020 40%),
linear-gradient(135deg,#050816,#0F172A);
color:white;
}

/* ========= SIDEBAR ========= */

section[data-testid="stSidebar"]{
background:rgba(15,23,42,.45);
backdrop-filter:blur(30px);
border-right:1px solid rgba(255,255,255,.08);
white-space: nowrap;
}

/* ========= HEADER ========= */

h1{
font-size:4rem!important;
font-weight:800!important;
color:white!important;
}

p{
color:#94A3B8!important;
}

/* ========= BUTTONS ========= */

.stButton>button{

background:
linear-gradient(
135deg,
rgba(124,58,237,.4),
rgba(59,130,246,.25)
);

border:none;
border-radius:20px;
color:white;
height:50px;
font-weight:600;

backdrop-filter:blur(20px);

}

/* ========= INPUT BOX ========= */

.stChatInputContainer{

background:
rgba(255,255,255,.04);

backdrop-filter:blur(20px);

border-radius:30px;

border:
1px solid rgba(255,255,255,.08);

}

/* ========= CHAT ========= */

[data-testid="stChatMessage"]{

background:
rgba(255,255,255,.04);

backdrop-filter:blur(20px);

border:
1px solid rgba(255,255,255,.06);

border-radius:30px;

padding:20px;

margin-bottom:15px;

}

/* ========= SCROLLBAR ========= */

::-webkit-scrollbar{
width:6px;
}

::-webkit-scrollbar-thumb{
background:#7C3AED;
border-radius:20px;
}

</style>
""",unsafe_allow_html=True)


# ---------------- Sidebar ----------------
with st.sidebar:

    st.title("ClaimIntel")
    st.caption("AI Insurance Assistant")

    st.divider()

    st.markdown("### ⚡ Model")
    st.success("Meta Llama 3.3 70B")

    st.markdown("### 📚 Vector Database")
    st.info("FAISS")

    st.markdown("### 🤖 Backend")
    st.info("NVIDIA API")

    st.divider()

    if st.button("🗑 Clear Chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

# ---------------- Main Header ----------------
st.title("ClaimIntel Copilot")
st.caption("AI-Powered Insurance Policy Assistant")

# ---------------- Session State ----------------
if "messages" not in st.session_state:
    st.session_state.messages = []

# ---------------- Display Previous Messages ----------------
for message in st.session_state.messages:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])

        if message["role"] == "assistant" and "pages" in message:

            st.caption(
                "📄 Sources: " +
                ", ".join(
                    [f"Page {page}" for page in message["pages"]]
                )
            )

# ---------------- Chat Input ----------------
prompt = st.chat_input(
    "Ask anything about your insurance policy..."
)

# ---------------- User Question ----------------
if prompt:

    # Save user message
    st.session_state.messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    # Display user message
    with st.chat_message("user"):
        st.markdown(prompt)

    # Build conversation memory
    chat_history = ""

    for msg in st.session_state.messages[-6:]:

        chat_history += (
            f"{msg['role']}: {msg['content']}\n"
        )

    # Generate response
    with st.chat_message("assistant"):

        with st.spinner("Analyzing policy documents..."):

            result = ask_claimintel(
                prompt,
                chat_history
            )

            answer = result["answer"]

            pages = result["pages"]

            st.markdown(answer)

            st.caption(
                "📄 Sources: " +
                ", ".join(
                    [f"Page {page}" for page in pages]
                )
            )

    # Save assistant response
    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": answer,
            "pages": pages
        }
    )

Overwriting app.py


In [58]:
!pkill streamlit
!streamlit run app.py &>/content/logs.txt &

In [59]:
!pip install pyngrok

In [60]:
from pyngrok import ngrok

ngrok.set_auth_token("3FPD0FBUspTZbAl1f1XHkZ8KxYk_63at5ciUhyNrmpzfaDLen")

ngrok.kill()

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://scary-acre-barman.ngrok-free.dev" -> "http://localhost:8501"
